In [1]:
import os
from pyspark.sql import SparkSession
from azure.storage.blob import BlobServiceClient


In [2]:
# Load .env
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
def get_blob_service_client():
    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    if not connection_string:
        raise ValueError("Cannot find azure connection string in env")
    print("Successfully get azure connection string from env:", connection_string)
    return BlobServiceClient.from_connection_string(connection_string)



In [4]:
def get_spark_session():
    storage_account = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
    account_key = os.getenv("AZURE_STORAGE_ACCOUNT_KEY")
    spark = (
        SparkSession.builder
        .master("local[*]")
        .appName("VN30 Stock Analysis")
        .config("spark.jars.packages", "org.apache.hadoop:hadoop-azure:3.4.1")
        .config("spark.sql.legacy.parquet.nanosAsLong", "true")
        .getOrCreate()
    )
    
    spark.conf.set(
        f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
        "SharedKey"
    )
    spark.conf.set(
        f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
        account_key
    )

    spark.conf.set("spark.sql.session.timeZone", "Asia/Bangkok")
    
    return spark

In [5]:
spark = get_spark_session()
storage_account = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")

from pyspark.sql.functions import input_file_name, regexp_extract, upper
from pyspark.sql import functions as F
df = (
    spark.read.parquet(
        f"abfss://raw@{storage_account}.dfs.core.windows.net/2026/03/28/*.parquet"
    )
    .withColumn("_source_file", input_file_name())
    .withColumn(
        "Ticker",
        upper(regexp_extract("_source_file", r"/([^/]+)\.parquet$", 1))
    )
    .withColumn("event_time", F.expr("timestamp_micros(CAST(Time / 1000 AS BIGINT))"))
    .drop("_source_file")
    .drop("Time")
    .select("Ticker", "Event_time", "Open", "High", "Low", "Close", "Volume")
)

df.show(truncate=False)
print(f"Total rows: {df.count()}")


AnalysisException: [PATH_NOT_FOUND] Path does not exist: abfss://raw@stvn30stock.dfs.core.windows.net/2026/03/28/*.parquet. SQLSTATE: 42K03

In [ ]:
# from vnstock import Vnstock
# stock = Vnstock().stock(symbol='ACB', source='KBS')

# # Lấy dữ liệu lịch sử giá cổ phiếu trong 1 năm qua
# df = stock.quote.history(start='2026-03-01', end='2026-03-29', interval='1D')
# print(df)